In [1]:
#Data Manipulation specific
import numpy as np
import pandas as pd
import xarray as xr
from netCDF4 import Dataset
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import copy as cp
import rasterio
from rasterio.transform import from_origin
import gc

from scipy.stats import ttest_ind


/home/anas.oubaha/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
!free -g

              total        used        free      shared  buff/cache   available
Mem:            376          61         154           7         159         304
Swap:             0           0           0


**Read NC data and extract 800 and 250 gphs**

In [3]:
# Load dataset
ds = xr.open_dataset('data/850&200PH_Div_80N20S50W80E.nc', engine="netcdf4")# Load NetCDF file
#ds = ds.sel(latitude=slice(45, 15))
#ds = ds.sel(valid_time=slice('1991-01-01', None))
ds = ds[['d']] #keep only the variable d

dsl = ds.where(ds.pressure_level.isin([850]), drop=True)
dsl = dsl.squeeze()
dsh = ds.where(ds.pressure_level.isin([200]), drop=True)
dsh = dsh.squeeze()

In [4]:
ds

<xarray.Dataset>
Dimensions:         (valid_time: 408, pressure_level: 2, latitude: 401,
                     longitude: 521)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 1991-01-01 ... 2024-12-01
  * pressure_level  (pressure_level) float64 850.0 200.0
  * latitude        (latitude) float64 80.0 79.75 79.5 ... -19.5 -19.75 -20.0
  * longitude       (longitude) float64 -50.0 -49.75 -49.5 ... 79.5 79.75 80.0
Data variables:
    d               (valid_time, pressure_level, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-06-10T12:16 GRIB to CDM+CF via cfgrib-0.9.1...

**Seasonal Div anomalies**

In [ ]:

# Helper function to assign season
def assign_season_and_year(df):
    df['year'] = df['valid_time'].dt.year
    df['month'] = df['valid_time'].dt.month
    month = df['month'].values

    # Assign seasons
    df['season'] = np.select(
        [
            np.isin(month, [12, 1, 2]),
            np.isin(month, [3, 4, 5]),
            np.isin(month, [6, 7, 8]),
            np.isin(month, [9, 10, 11])
        ],
        ['DJF', 'MAM', 'JJA', 'SON']
    )

    # Adjust season year (Dec belongs to next year)
    df['season_year'] = df['year'].values
    df.loc[month == 12, 'season_year'] += 1

    return df

# Prepare list to collect seasonal stats
seasonal_results = []

# Set chunk size (you can increase or decrease based on available memory)
chunk_size = 100

# Loop through time chunks
for start in range(0, dsh.dims['valid_time'], chunk_size):
    end = min(start + chunk_size, dsh.dims['valid_time'])

    # Subset dataset
    subset = dsh.isel(valid_time=slice(start, end))

    # Convert to DataFrame with coordinates
    df = subset['d'].to_dataframe().dropna().reset_index()

    # Assign season and season year
    df = assign_season_and_year(df)

    # Compute seasonal average for this chunk
    seasonal_avg = (
        df.groupby(['latitude', 'longitude', 'season_year', 'season'], sort=False)['d']
        .mean()
        .reset_index()
    )

    seasonal_results.append(seasonal_avg)

    # Clean up memory
    del df, subset, seasonal_avg
    gc.collect()

# Concatenate all chunk results into full seasonal dataframe
seasonal_zl_df = pd.concat(seasonal_results, ignore_index=True)

# Compute 1991–2020 climatology
filtered = seasonal_zl_df.query("1991 <= season_year <= 2020")
seasonal_climatology_df = (
    filtered.groupby(['latitude', 'longitude', 'season'], sort=False)['d']
    .mean()
    .reset_index()
    .rename(columns={'d': 'zl_climatology'})
)

# Merge actual values with climatology
mergedf = pd.merge(
    seasonal_zl_df,
    seasonal_climatology_df,
    on=['latitude', 'longitude', 'season'],
    how='left'
)

# Compute anomalies
mergedf['zl_anomaly'] = mergedf['d'] - mergedf['zl_climatology']

# Optional: Add readable season-year label
mergedf['date'] = mergedf['season'] + ' ' + mergedf['season_year'].astype(str)

# Final output
mergedf.head()


In [ ]:

# Convert 'date' from integer to datetime format
dsh = dsh.assign_coords(date=pd.to_datetime(dsh['valid_time'].data.astype(str), format='ISO8601'))

# Convert dataset to DataFrame
df = dsh['d'].to_dataframe().reset_index()

# Extract year and month
df['year'] = df['valid_time'].dt.year
df['month'] = df['valid_time'].dt.month

# **FAST NumPy Vectorized Method for Assigning Seasons**
seasons = np.array(["DJF", "MAM", "JJA", "SON"])

# Define conditions for season assignment
winter = df['month'].isin([12, 1, 2])
spring = df['month'].isin([3, 4, 5])
summer = df['month'].isin([6, 7, 8])
autumn = df['month'].isin([9, 10, 11])

# Assign seasons using np.select (much faster than apply)
df['season'] = np.select([winter, spring, summer, autumn], seasons)

# Assign season_year (December belongs to next year’s winter)
df['season_year'] = df['year']
df.loc[df['month'] == 12, 'season_year'] += 1  # Shift December to next year's winter

# **STEP 1: Compute Actual Seasonal SST (For Each Year)**
seasonal_zl_df = df.groupby(['latitude', 'longitude', 'season_year', 'season'])['d'].mean().reset_index()

# **STEP 2: Compute Seasonal Climatology (Long-Term Average Over All Years)**
# Filter to only include seasons from 1991 to 2020 (30 years period recommended by the WMO)
filtered_df = seasonal_zl_df[(seasonal_zl_df['season_year'] >= 1991) & (seasonal_zl_df['season_year'] <= 2020)]

seasonal_climatology_df = filtered_df.groupby(['latitude', 'longitude', 'season'])['d'].mean().reset_index()
seasonal_climatology_df.rename(columns={'d': 'zl_climatology'}, inplace=True)

# **STEP 3: Merge Actual SST With Climatology**
mergedf = pd.merge(seasonal_zl_df, seasonal_climatology_df, on=['latitude', 'longitude', 'season'], how="left")
# **STEP 4: Compute Seasonal Anomalies**
mergedf['zl_anomaly'] = mergedf['d'] - mergedf['zl_climatology']

mergedf['date'] = (
    mergedf['season'].astype(str) 
    + ' ' 
    + mergedf['season_year'].astype(str)
)
mergedf

**Apply Welch’s T-Test**

In [9]:
from scipy.stats import ttest_ind

# Define drought years
drought_years = [1994, 1995, 1999, 2000, 2001, 2005, 2016, 2019, 2020, 2021, 2022, 2023]  # Replace with your own

# Create a unique season key
#mergedf['season_key'] = mergedf['season'] + '_' + mergedf['season_year'].astype(str)

# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedf.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='zl_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge with original mergedf
mergedf = pd.merge(mergedf, p_df, on=['latitude', 'longitude', 'season'], how='left')
mergedf

,latitude,longitude,season_year,season,z,zl_climatology,zl_anomaly,date,p_value
0,-20.0,-50.0,1991,DJF,12437.819336,12425.171875,12.647461,DJF 1991,0.461277
1,-20.0,-50.0,1991,JJA,12321.921875,12324.198242,-2.276367,JJA 1991,0.817325
2,-20.0,-50.0,1991,MAM,12392.734375,12390.902344,1.832031,MAM 1991,0.842538
3,-20.0,-50.0,1991,SON,12338.250977,12371.598633,-33.347656,SON 1991,0.803617
4,-20.0,-50.0,1992,DJF,12410.333008,12425.171875,-14.838867,DJF 1992,0.461277
...,...,...,...,...,...,...,...,...,...
28622172,80.0,80.0,2024,DJF,10982.229492,10925.591797,56.637695,DJF 2024,0.385162
28622173,80.0,80.0,2024,JJA,11785.278320,11694.649414,90.628906,JJA 2024,0.561397
28622174,80.0,80.0,2024,MAM,11241.584961,11185.811523,55.773438,MAM 2024,0.666898
28622175,80.0,80.0,2024,SON,11276.796875,11238.290039,38.506836,SON 2024,0.916939


In [10]:
# Convert back to xarray
anomalies = mergedf.set_index(['date', 'latitude', 'longitude'])['zl_anomaly'].to_xarray()
pval = mergedf.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()
anomalies

<xarray.DataArray 'zl_anomaly' (date: 137, latitude: 401, longitude: 521)>
array([[[  12.647461 ,   12.375    ,   12.1015625, ...,   18.49707  ,
           18.611328 ,   18.691406 ],
        [  12.97168  ,   12.693359 ,   12.438477 , ...,   17.733398 ,
           17.832031 ,   17.89746  ],
        [  13.268555 ,   12.996094 ,   12.724609 , ...,   16.963867 ,
           17.044922 ,   17.104492 ],
        ...,
        [ -49.70117  ,  -49.411133 ,  -49.11035  , ...,   -2.9853516,
           -3.0302734,   -3.0576172],
        [ -50.160156 ,  -49.862305 ,  -49.5791   , ...,   -2.3916016,
           -2.4013672,   -2.4335938],
        [ -50.56836  ,  -50.29883  ,  -50.001953 , ...,   -1.8144531,
           -1.8408203,   -1.8720703]],

       [[ -14.838867 ,  -15.051758 ,  -15.267578 , ...,  -25.871094 ,
          -26.011719 ,  -26.186523 ],
        [ -14.978516 ,  -15.1796875,  -15.350586 , ...,  -25.549805 ,
          -25.72754  ,  -25.886719 ],
        [ -15.050781 ,  -15.238281 ,  -15.40332  , ...,  -25.214844 ,
          -25.39746  ,  -25.580078 ],
...
        [  40.856445 ,   40.791992 ,   40.69336  , ...,   28.458008 ,
           28.492188 ,   28.501953 ],
        [  40.342773 ,   40.268555 ,   40.198242 , ...,   27.585938 ,
           27.605469 ,   27.626953 ],
        [  39.83789  ,   39.782227 ,   39.70117  , ...,   26.762695 ,
           26.77539  ,   26.78418  ]],

       [[  57.273438 ,   57.155273 ,   57.009766 , ...,   66.29199  ,
           66.42285  ,   66.56152  ],
        [  56.836914 ,   56.72168  ,   56.60254  , ...,   65.69629  ,
           65.81738  ,   65.93652  ],
        [  56.4375   ,   56.302734 ,   56.20703  , ...,   65.09961  ,
           65.19922  ,   65.30957  ],
        ...,
        [  66.836914 ,   66.586914 ,   66.2832   , ...,   39.041992 ,
           38.88965  ,   38.710938 ],
        [  66.3418   ,   66.081055 ,   65.80469  , ...,   38.90039  ,
           38.76758  ,   38.60254  ],
        [  65.90527  ,   65.66211  ,   65.393555 , ...,   38.825195 ,
           38.668945 ,   38.506836 ]]], dtype=float32)
Coordinates:
  * date       (date) object 'DJF 1991' 'DJF 1992' ... 'SON 2023' 'SON 2024'
  * latitude   (latitude) float64 -20.0 -19.75 -19.5 -19.25 ... 79.5 79.75 80.0
  * longitude  (longitude) float64 -50.0 -49.75 -49.5 -49.25 ... 79.5 79.75 80.0

In [8]:
str(anomalies['date'][136].values)

'SON 2024'

In [11]:
import os
def export_to_geotiff(path, u_data, v_data, lat, lon, description):
    # Flip latitude if needed (GeoTIFF expects top-to-bottom order)
    if lat[0] > lat[-1]:
        u_data = u_data[::-1]
        v_data = v_data[::-1]
        lat = lat[::-1]

    # Get transform
    pixel_size_x = abs((lon[-1] - lon[0])) / (len(lon) - 1)
    pixel_size_y = abs((lat[-1] - lat[0])) / (len(lat) - 1)
    transform = from_origin(lon[0] - pixel_size_x / 2, lat[0] - pixel_size_y / 2, pixel_size_x, -pixel_size_y)

    # Stack u and v into a 3D array (bands, height, width)
    data = np.stack([u_data, v_data])  # shape: (2, lat, lon)

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=u_data.shape[0],
        width=u_data.shape[1],
        count=2,  # 2 bands: u and v
        dtype=u_data.dtype,
        crs="EPSG:4326",
        transform=transform,
        description=description
    ) as dst:
        dst.write(data[0], 1)  # Band 1 - u
        dst.write(data[1], 2)  # Band 2 - v
        
# Export regression slope map
output_dir = 'GPH200'  # no leading slash unless it's an absolute path
os.makedirs(output_dir, exist_ok=True)

for d in range(137):
    file_path = os.path.join(output_dir, str(anomalies['date'][d].values) + '.tif')
    export_to_geotiff(file_path, anomalies[d], pval[d], anomalies.latitude.values, anomalies.longitude.values, 'GPH200 Seasonal Anomalies of '+str(anomalies['date'][d].values))
